## Read direct XBPM measurement pickle files

### Read files, extract nominal and measured positions
#### Average positions and calculate their respective errors.
#### Plot nominal and measured positions.

In [81]:
def extract_positions(data) -> list:
    return [
        (np.mean(data['X_pos']), np.std(data['X_pos'])),
        (np.mean(data['Y_pos']), np.std(data['Y_pos']))
        ]

In [144]:
import pickle
import re
import numpy as np
from pathlib import Path

# Working directory.
# wdir = ("/home/arnaldo.filho/XBPM/Measurements/Analysis/2026/2026-10-26"
#         + "/data/xbpm_meas")

i = 2
wdir = ("/home/arnaldo.filho/XBPM/Measurements/Analysis/2026/2026-09-28/"
        + f"MNC/MNC_pass0{i}/data/xbpm_meas_pass0{i}")

target_dir = Path(wdir)

# Read file names.
pfiles = [
    p.name
    for p in target_dir.iterdir()
    if (
        p.is_file() and
        p.name.startswith("xbpm_acq_") and
        p.name.endswith(".pickle")
        )
    ]


# Assemble MNC 1 & 2 data.
mnc1 = {}
mnc2 = {}

for pf in pfiles:

    # Read the angles from filename.
    agx = int(re.search(r"agx:([-\d]+)", pf).group(1))
    agy = int(re.search(r"agy:([-\d]+)", pf).group(1))
    print(f"Processing file with angles: ({agx:>7.2f},  {agy:>7.2f}) ")

    filename = f"{target_dir}/{pf}"
    with open(filename, "rb") as f:
        xbpm = pickle.load(f)
        mnc1[(agx, agy)] = extract_positions(xbpm['MNC1'])
        mnc2[(agx, agy)] = extract_positions(xbpm['MNC2'])

        # print(f"\n###\n### KEYS:\n###\n {xbpm.keys()}\n")

        # print("\n###\n### Data:\n###\n\n")
        # for k, d in xbpm['MNC1'].items():
        #     print(f"{k}: {d}")


Processing file with angles: (   5.00,     0.00) 
Processing file with angles: (  10.00,     0.00) 
Processing file with angles: ( -10.00,     0.00) 
Processing file with angles: (   0.00,     5.00) 
Processing file with angles: ( -10.00,    10.00) 
Processing file with angles: (  -5.00,     0.00) 
Processing file with angles: (  10.00,   -10.00) 
Processing file with angles: (  -5.00,    10.00) 
Processing file with angles: (   0.00,   -10.00) 
Processing file with angles: (  10.00,    10.00) 
Processing file with angles: (   5.00,    10.00) 
Processing file with angles: (  -5.00,   -10.00) 
Processing file with angles: (   0.00,     0.00) 
Processing file with angles: (   0.00,    10.00) 
Processing file with angles: (   5.00,     5.00) 
Processing file with angles: (   0.00,    -5.00) 
Processing file with angles: ( -10.00,   -10.00) 
Processing file with angles: ( -10.00,    -5.00) 
Processing file with angles: (  10.00,     5.00) 
Processing file with angles: (   5.00,    -5.00) 



### Put data into numpy array structures.

In [145]:
unitfactor = 1  #  e-3    # nm to µm
xbpmdist_1 = 15.740  # m
xbpmdist_2 = 19.590  # m

angles = sorted(mnc1.keys())

# MNC 1 
xn1 = np.array([k[0] for k in angles]) * xbpmdist_1
yn1 = np.array([k[1] for k in angles]) * xbpmdist_1

xp1 = np.array([mnc1[k][0][0] for k in angles]) * unitfactor
yp1 = np.array([mnc1[k][1][0] for k in angles]) * unitfactor

offset_x = mnc1[0, 0][0][0] * unitfactor
offset_y = mnc1[0, 0][1][0] * unitfactor
xp1 -= offset_x
yp1 -= offset_y

sxp1 = np.array([mnc1[k][0][1] for k in angles]) * unitfactor
syp1 = np.array([mnc1[k][1][1] for k in angles]) * unitfactor

# MNC 2
xn2 = np.array([k[0] for k in angles]) * xbpmdist_2
yn2 = np.array([k[1] for k in angles]) * xbpmdist_2

xp2 = np.array([mnc2[k][0][0] for k in angles]) * unitfactor
yp2 = np.array([mnc2[k][1][0] for k in angles]) * unitfactor

offset_x = mnc2[0, 0][0][0] * unitfactor
offset_y = mnc2[0, 0][1][0] * unitfactor
xp2 -= offset_x
yp2 -= offset_y

sxp2 = np.array([mnc2[k][0][1] for k in angles]) * unitfactor
syp2 = np.array([mnc2[k][1][1] for k in angles]) * unitfactor


In [146]:
%matplotlib qt5

In [152]:
len(mnc1)

25

### Plot nominal and XBPM-measured grids.

In [153]:
import matplotlib.pyplot as plt

# plt.plot(xn1, yn1, 'r+')

fig, ((ax1, ax2), (axdiff1,  axdiff2)) = plt.subplots(2, 2, figsize=(12, 6))

# Reference positions.
ax1.plot(xn1, yn1, 'r+')
ax2.plot(xn2, yn2, 'r+')

rsize = 5

ax1.errorbar(xp1, yp1, xerr=sxp1, yerr=syp1,
             fmt='o', color='blue',
             ecolor='lightblue', elinewidth=2,
             capsize=4)

ax2.errorbar(xp2, yp2, xerr=sxp2, yerr=syp2,
             fmt='o', color='darkgreen',
             ecolor='grey', elinewidth=2,
             capsize=4)

diff1 = np.sqrt(
    (xp1.reshape(rsize, rsize) - xn1.reshape(rsize, rsize))**2 +
    (yp1.reshape(rsize, rsize) - yn1.reshape(rsize, rsize))**2
    )
extent = [xn1.min(), xn1.max(), yn1.min(), yn1.max()]
axdiff1.imshow(diff1, cmap='viridis', aspect='auto', extent=extent)
axdiff1.set_title('Difference')
axdiff1.set_xlabel('x [µm]')
axdiff1.set_ylabel('y [µm]')

diff2 = np.sqrt(
    (xp2.reshape(rsize, rsize) - xn2.reshape(rsize, rsize))**2 +
    (yp2.reshape(rsize, rsize) - yn2.reshape(rsize, rsize))**2
    )
axdiff2.imshow(diff2, cmap='viridis', aspect='auto', extent=extent)
axdiff2.set_title('Difference')
axdiff2.set_xlabel('x [µm]')
axdiff2.set_ylabel('y [µm]')
cbar1 = fig.colorbar(axdiff1.imshow(diff1, cmap='viridis', aspect='auto', extent=extent), ax=axdiff1)
cbar1.set_label('Difference [µm]')
cbar2 = fig.colorbar(axdiff2.imshow(diff2, cmap='viridis', aspect='auto', extent=extent), ax=axdiff2)
cbar2.set_label('Difference [µm]')

fig.tight_layout()
ax1.set_xlabel('x [µm]')
ax1.set_ylabel('y [µm]')
ax2.set_xlabel('x [µm]')
ax2.set_ylabel('y [µm]')
ax1.grid(True)
ax2.grid(True)
axdiff1.grid(True)
axdiff2.grid(True)

plt.show()